In [ ]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install stanza
import stanza

# download both Hindi and English models
stanza.download('hi')  # Hindi model
stanza.download('en')  # English model

import pandas as pd
import numpy as np
import re
import os
import json
from tqdm import tqdm

DRIVE_BASE = "/content/drive/MyDrive/Transcripts_CSS"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "outputs", "stanza_annotations")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Setup complete")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 85.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 794.2/794.2 kB 57.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 608.4/608.4 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 418.7/418.7 kB 40.3 MB/s eta 0:00:00


INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Downloading default packages for language: hi (Hindi) ...


models/default.zip: reconstructing file:   0%|          |  0.00B /  320MB            

models/default.zip: downloading bytes:           |  0.00B            

INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/hi/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.14.0/resources


INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/resources.json
INFO:stanza:Downloading default packages for language: en (English) ...


models/default.zip: reconstructing file:   0%|          |  0.00B /  525MB            

models/default.zip: downloading bytes:           |  0.00B            

INFO:stanza:Downloaded file to /root/.cache/stanza/1.14.0/resources/en/default.zip
INFO:stanza:Finished downloading models and saved to /root/.cache/stanza/1.14.0/resources


Setup complete


Cell 2 — Load Stanza Pipelines

In [ ]:
# Hindi pipeline — tokenization, POS, lemma, dependency parse
nlp_hi = stanza.Pipeline(
    'hi',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=True,
    verbose=False
)

# English pipeline — same processors + NER
nlp_en = stanza.Pipeline(
    'en',
    processors='tokenize,pos,lemma,depparse',
    use_gpu=True,
    verbose=False
)

print("Both pipelines loaded")

Both pipelines loaded


Cell 3 — Script Detection Utilities

In [ ]:
def detect_script_ratio(text):
    """
    Returns ratio of Devanagari vs Roman characters.
    Used to decide which Stanza model to apply.
    """
    if not text or len(text.strip()) == 0:
        return 0.0, 0.0

    devanagari = len(re.findall(r'[\u0900-\u097F]', text))
    roman = len(re.findall(r'[a-zA-Z]', text))
    total = devanagari + roman

    if total == 0:
        return 0.0, 0.0

    return devanagari / total, roman / total


def classify_segment_language(text, threshold=0.4):
    """
    Classify a text segment as:
    - 'hi'      : predominantly Devanagari
    - 'en'      : predominantly Roman/English
    - 'mixed'   : genuinely mixed Hinglish
    """
    hi_ratio, en_ratio = detect_script_ratio(text)

    if hi_ratio >= threshold:
        return 'hi'
    elif en_ratio >= threshold:
        return 'en'
    else:
        return 'mixed'  # short or ambiguous segments

Cell 4 — Disfluency Detection

In [ ]:
# disfluency patterns for Hindi/Hinglish speech
# these are what you want to detect and optionally remove

DISFLUENCY_PATTERNS = {

    # English fillers
    'filler_en': re.compile(
        r'\b(uh+|um+|hmm+|hm+|ah+|oh+|er+|erm+)\b',
        re.IGNORECASE
    ),

    # English discourse markers
    'discourse_en': re.compile(
        r'\b(like|basically|actually|literally|obviously|'
        r'you know|i mean|right|okay|so|well|anyway|'
        r'kind of|sort of|you see|i guess|i think)\b',
        re.IGNORECASE
    ),

    # Hindi fillers — Devanagari
    'filler_hi_dev': re.compile(
        r'\b(अं|उं|हं|आं|ओह|अरे|हाँ|यार)\b'
    ),

    # Hindi discourse markers — Devanagari
    'discourse_hi_dev': re.compile(
        r'\b(मतलब|देखो|देख|समझो|बस|तो|वैसे|'
        r'अच्छा|सही है|ठीक है|यानी|जैसे)\b'
    ),

    # Romanized Hindi fillers
    'filler_hi_rom': re.compile(
        r'\b(arrey|arre|matlab|haan|haa|yaar|'
        r'achha|theek|bas|toh|waise)\b',
        re.IGNORECASE
    ),

    # False starts — repeated words/phrases
    'false_start': re.compile(
        r'\b(\w+)\s+\1\b',  # word immediately repeated
        re.IGNORECASE
    ),

    # Incomplete words with dashes
    'incomplete': re.compile(r'\b\w+-\s'),
}


def detect_disfluencies(text):
    """
    Find all disfluencies in a text segment.
    Returns a dict of disfluency type → list of matches.
    """
    results = {}
    for disfluency_type, pattern in DISFLUENCY_PATTERNS.items():
        matches = pattern.findall(text)
        if matches:
            results[disfluency_type] = matches
    return results


def flag_discourse_markers(text):
    """
    Specifically flag discourse markers (NOT remove them —
    these are your GDCF targets, so you want to keep them
    but mark them clearly in the annotation).
    Returns list of found discourse markers.
    """
    markers = []

    # English discourse markers
    en_matches = DISFLUENCY_PATTERNS['discourse_en'].findall(text)
    markers.extend([('en', m) for m in en_matches])

    # Hindi discourse markers — Devanagari
    hi_dev_matches = DISFLUENCY_PATTERNS['discourse_hi_dev'].findall(text)
    markers.extend([('hi_dev', m) for m in hi_dev_matches])

    # Hindi discourse markers — Romanized
    hi_rom_matches = DISFLUENCY_PATTERNS['filler_hi_rom'].findall(text)
    markers.extend([('hi_rom', m) for m in hi_rom_matches])

    return markers

Cell 5 — Stanza Annotation Function

In [ ]:

def annotate_with_stanza(text, language):
    if not text or len(text.strip()) < 3:
        return []

    annotations = []
    # always split into sentences and route each one individually
    sentences = re.split(r'[।\.!?]+', text)

    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        # detect language per sentence — not inherited from document
        sent_lang = classify_segment_language(sent)
        if sent_lang == 'mixed':
            hi_r, en_r = detect_script_ratio(sent)
            sent_lang = 'hi' if hi_r >= en_r else 'en'

        nlp = nlp_hi if sent_lang == 'hi' else nlp_en
        doc = nlp(sent)

        for sentence in doc.sentences:
            for word in sentence.words:
                annotations.append({
                    'text': word.text,
                    'lemma': word.lemma,
                    'upos': word.upos,
                    'xpos': word.xpos,
                    'deprel': word.deprel,
                    'head': word.head,
                    'feats': word.feats
                })

    return annotations


def get_pos_summary(annotations):
    """
    Summarize POS distribution for a segment.
    Useful for checking discourse vs content ratio.
    """
    if not annotations:
        return {}

    from collections import Counter
    pos_counts = Counter([a['upos'] for a in annotations])
    total = len(annotations)
    return {pos: round(count/total, 3) for pos, count in pos_counts.items()}

Cell 6 — Fluency Cleaning Function

In [ ]:
def remove_false_starts_safe(text):
    words = text.split(' ')
    out = []
    prev = None
    for w in words:
        if w == prev:
            continue
        out.append(w)
        prev = w
    return ' '.join(out)

def clean_transcript(text, remove_fillers=True, remove_false_starts=True):
    """
    Produce a cleaner, more fluent version of a transcript.

    IMPORTANT: This does NOT remove discourse markers
    (like, basically, mतलब etc.) — those are kept because
    they are your GDCF targets. Only true fillers and
    false starts are removed.
    """
    cleaned = text

    if remove_fillers:
        # remove pure fillers (uh, um, hmm, arrey etc.)
        cleaned = DISFLUENCY_PATTERNS['filler_en'].sub('', cleaned)
        cleaned = DISFLUENCY_PATTERNS['filler_hi_dev'].sub('', cleaned)
        # NOTE: filler_hi_rom overlaps with discourse markers
        # so we do NOT apply filler_hi_rom removal here
        # to avoid removing your GDCF targets

    if remove_false_starts:
        cleaned = remove_false_starts_safe(cleaned)          # replaces the regex line
        cleaned = DISFLUENCY_PATTERNS['incomplete'].sub('', cleaned)  # dash-based, check this one too

    # clean up extra whitespace
    cleaned = re.sub(r'\s+', ' ', cleaned).strip()

    return cleaned

Cell 7 — Main Annotation Pipeline Per Document

In [ ]:
def process_transcript(video_id, transcript, community, creator_gender):
    """
    Full annotation pipeline for a single transcript.
    Returns structured annotation dict.
    """

    if not transcript or len(transcript.strip()) < 10:
        return None

    # --- Step 1: Script detection ---
    language = classify_segment_language(transcript)
    hi_ratio, en_ratio = detect_script_ratio(transcript)

    # --- Step 2: Disfluency detection ---
    disfluencies = detect_disfluencies(transcript)
    discourse_markers = flag_discourse_markers(transcript)

    # --- Step 3: Stanza annotation (always sentence-level routing) ---
    sentences = re.split(r'[।\.!?]+', transcript)
    all_annotations = []
    for sent in sentences:
        sent = sent.strip()
        if not sent:
            continue
        sent_lang = classify_segment_language(sent)
        if sent_lang == 'mixed':
            sent_lang = 'hi' if hi_ratio >= en_ratio else 'en'
        sent_annotations = annotate_with_stanza(sent, sent_lang)
        all_annotations.extend(sent_annotations)

    # --- Step 4: Clean transcript ---
    cleaned = clean_transcript(transcript)

    # --- Step 5: POS summary ---
    pos_summary = get_pos_summary(all_annotations)

    return {
        'video_id': video_id,
        'community': community,
        'creator_gender': creator_gender,
        'language_detected': language,
        'hi_script_ratio': round(hi_ratio, 3),
        'en_script_ratio': round(en_ratio, 3),
        'transcript_original': transcript,
        'transcript_cleaned': cleaned,
        'n_tokens_original': len(transcript.split()),
        'n_tokens_cleaned': len(cleaned.split()),
        'disfluencies_found': disfluencies,
        'discourse_markers_found': discourse_markers,
        'n_discourse_markers': len(discourse_markers),
        'stanza_annotations': all_annotations,
        'pos_summary': pos_summary,
        'n_tokens_annotated': len(all_annotations)
    }

Cell 8 — Run on Your DataFrame

In [ ]:
# ============================================================
# PROCESS FULL DATASET WITH CHECKPOINT SAVING
# ============================================================

import os
import json
import pandas as pd
from tqdm.auto import tqdm

# ------------------------------------------------------------
# 1. Load df
# ------------------------------------------------------------
df_path = os.path.join(DRIVE_BASE, "outputs", "df_transcripts_religion.csv")

df = pd.read_csv(df_path)
df["transcript"] = df["transcript"].fillna("")

print(f"Loaded {len(df):,} transcripts from df_transcripts_religion.csv")

# ------------------------------------------------------------
# 2. Checkpoint settings
# ------------------------------------------------------------
CHECKPOINT_EVERY = 100

checkpoint_dir = os.path.join(
    OUTPUT_DIR,
    "stanza_checkpoints"
)
os.makedirs(checkpoint_dir, exist_ok=True)

# ------------------------------------------------------------
# 3. Process transcripts
# ------------------------------------------------------------
results = []
failed = []

for idx, row in tqdm(
    df.iterrows(),
    total=len(df),
    desc="Processing transcripts"
):

    video_id = row.get("video_id", f"video_{idx}")

    try:
        result = process_transcript(
            video_id=video_id,
            transcript=row["transcript"],
            community=row.get("community", "unknown"),
            creator_gender=row.get("creator_gender", "unknown")
        )

        if result:
            results.append(result)

    except Exception as e:
        failed.append({
            "idx": idx,
            "video_id": video_id,
            "error": str(e)
        })

        print(f"❌ Failed: {video_id} — {e}")

    # --------------------------------------------------------
    # CHECKPOINT
    # --------------------------------------------------------
    if (idx + 1) % CHECKPOINT_EVERY == 0:

        checkpoint_results_path = os.path.join(
            checkpoint_dir,
            f"results_{idx + 1}.json"
        )

        checkpoint_failed_path = os.path.join(
            checkpoint_dir,
            f"failed_{idx + 1}.json"
        )

        with open(
            checkpoint_results_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                results,
                f,
                ensure_ascii=False
            )

        with open(
            checkpoint_failed_path,
            "w",
            encoding="utf-8"
        ) as f:
            json.dump(
                failed,
                f,
                ensure_ascii=False
            )

        print(
            f"\n💾 Checkpoint saved at {idx + 1:,} transcripts "
            f"| successful: {len(results):,} "
            f"| failed: {len(failed):,}"
        )


# ------------------------------------------------------------
# 4. Final checkpoint
# ------------------------------------------------------------

final_results_path = os.path.join(
    checkpoint_dir,
    "results_FINAL.json"
)

final_failed_path = os.path.join(
    checkpoint_dir,
    "failed_FINAL.json"
)

with open(final_results_path, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False)

with open(final_failed_path, "w", encoding="utf-8") as f:
    json.dump(failed, f, ensure_ascii=False)

print("\n" + "=" * 60)
print("PROCESSING COMPLETE")
print("=" * 60)

print(f"Total transcripts:      {len(df):,}")
print(f"Successfully processed: {len(results):,}")
print(f"Failed:                 {len(failed):,}")
print(f"Checkpoint directory:   {checkpoint_dir}")


Loaded 1,033 transcripts from df_transcripts_religion.csv


Processing transcripts:   0%|          | 0/1033 [00:00<?, ?it/s]


💾 Checkpoint saved at 100 transcripts | successful: 100 | failed: 0

💾 Checkpoint saved at 200 transcripts | successful: 200 | failed: 0

💾 Checkpoint saved at 300 transcripts | successful: 300 | failed: 0

💾 Checkpoint saved at 400 transcripts | successful: 400 | failed: 0

💾 Checkpoint saved at 500 transcripts | successful: 500 | failed: 0

💾 Checkpoint saved at 600 transcripts | successful: 600 | failed: 0

💾 Checkpoint saved at 700 transcripts | successful: 700 | failed: 0

💾 Checkpoint saved at 800 transcripts | successful: 800 | failed: 0

💾 Checkpoint saved at 900 transcripts | successful: 900 | failed: 0

💾 Checkpoint saved at 1,000 transcripts | successful: 1,000 | failed: 0

PROCESSING COMPLETE
Total transcripts:      1,033
Successfully processed: 1,033
Failed:                 0
Checkpoint directory:   /content/drive/MyDrive/Transcripts_CSS/outputs/stanza_annotations/stanza_checkpoints


Cell 9 — Save Outputs

In [ ]:
# --- Save 1: Lightweight summary CSV (no full annotations) ---
summary_records = []

for r in results:
    summary_records.append({
        'video_id': r['video_id'],
        'community': r['community'],
        'creator_gender': r['creator_gender'],
        'language_detected': r['language_detected'],
        'hi_script_ratio': r['hi_script_ratio'],
        'en_script_ratio': r['en_script_ratio'],
        'n_tokens_original': r['n_tokens_original'],
        'n_tokens_cleaned': r['n_tokens_cleaned'],
        'n_discourse_markers': r['n_discourse_markers'],
        'n_tokens_annotated': r['n_tokens_annotated'],
        'transcript_cleaned': r['transcript_cleaned'],
        'pos_summary': json.dumps(
            r['pos_summary'],
            ensure_ascii=False
        ),
        'discourse_markers_found': json.dumps(
            r['discourse_markers_found'],
            ensure_ascii=False
        ),
    })

summary_df = pd.DataFrame(summary_records)

summary_path = os.path.join(
    OUTPUT_DIR,
    "stanza_religion_summary.csv"
)

summary_df.to_csv(summary_path, index=False)

print(f"Summary CSV saved: {summary_path}")


# --- Save 2: Full annotations as JSON ---
full_path = os.path.join(
    OUTPUT_DIR,
    "stanza_religion_full_annotations.json"
)

with open(full_path, "w", encoding="utf-8") as f:
    json.dump(
        results,
        f,
        ensure_ascii=False,
        indent=2
    )

print(f"Full annotations saved: {full_path}")


# --- Save 3: Failed videos log ---
if failed:
    failed_path = os.path.join(
        OUTPUT_DIR,
        "stanza_religion_failed.json"
    )

    with open(failed_path, "w", encoding="utf-8") as f:
        json.dump(
            failed,
            f,
            ensure_ascii=False,
            indent=2
        )

    print(f"Failed log saved: {failed_path}")


Summary CSV saved: /content/drive/MyDrive/Transcripts_CSS/outputs/stanza_annotations/stanza_religion_summary.csv
Full annotations saved: /content/drive/MyDrive/Transcripts_CSS/outputs/stanza_annotations/stanza_religion_full_annotations.json


Cell 10 — Quick Validation Check

In [ ]:
sample = results[0]

print(f"Video: {sample['video_id']}")
print(f"Community: {sample['community']}")
print(f"Language detected: {sample['language_detected']}")
print(f"Hindi ratio: {sample['hi_script_ratio']}")
print(f"English ratio: {sample['en_script_ratio']}")
print(f"Discourse markers found: {sample['discourse_markers_found'][:5]}")
print(f"POS summary: {sample['pos_summary']}")

print(
    f"\nOriginal (first 200 chars):\n"
    f"{sample['transcript_original'][:200]}"
)

print(
    f"\nCleaned (first 200 chars):\n"
    f"{sample['transcript_cleaned'][:200]}"
)

print("\nFirst 5 token annotations:")

for tok in sample['stanza_annotations'][:5]:
    print(
        f"  {tok['text']:15} | "
        f"POS: {tok['upos']:6} | "
        f"Lemma: {tok['lemma']:15} | "
        f"Dep: {tok['deprel']}"
    )


Video: AnandmurtiGurumaa__zGI-5AUoev0.mp3
Community: transcripts_religion
Language detected: hi
Hindi ratio: 0.996
English ratio: 0.004
Discourse markers found: [('hi_dev', 'देख'), ('hi_dev', 'देख'), ('hi_dev', 'बस')]
POS summary: {'DET': 0.051, 'PRON': 0.146, 'NOUN': 0.191, 'VERB': 0.143, 'AUX': 0.134, 'PART': 0.051, 'CCONJ': 0.024, 'ADP': 0.107, 'ADJ': 0.039, 'PUNCT': 0.051, 'NUM': 0.013, 'ADV': 0.014, 'SCONJ': 0.023, 'PROPN': 0.012}

Original (first 200 chars):
कितने किसके कब-कब जन्म हो चुके है कोई जीव है जो सिर्फ विशेवासना और भोग के साथ अपने जीवन को बिता देता है कोई है जो धन को इकत्र करने में कोई है जो शक्तियां कठी कर लूँ, राजनेतिक सक्ता को प्राप्त कर लूँ व

Cleaned (first 200 chars):
कितने किसके कब-कब जन्म हो चुके है कोई जीव है जो सिर्फ विशेवासना और भोग के साथ अपने जीवन को बिता देता है कोई है जो धन को इकत्र करने में कोई है जो शक्तियां कठी कर लूँ, राजनेतिक सक्ता को प्राप्त कर लूँ व

First 5 token annotations:
  कितने           | POS: DET    | Lemma: कितना           | Dep: det
  किस

In [ ]:
print("Input rows:", len(df))
print("Results:", len(results))
print("Failed:", len(failed))

if results:
    print("\nFirst result keys:")
    print(results[0].keys())

print("\nSummary dataframe shape:")
print(summary_df.shape)


Input rows: 1033
Results: 1033
Failed: 0

First result keys:
dict_keys(['video_id', 'community', 'creator_gender', 'language_detected', 'hi_script_ratio', 'en_script_ratio', 'transcript_original', 'transcript_cleaned', 'n_tokens_original', 'n_tokens_cleaned', 'disfluencies_found', 'discourse_markers_found', 'n_discourse_markers', 'stanza_annotations', 'pos_summary', 'n_tokens_annotated'])

Summary dataframe shape:
(1033, 13)
